# Data Collection
**Project:** Impact of Global Crises on Financial Markets


In [ ]:
# Install libraries (run this cell only once)
!pip install yfinance fredapi pandas matplotlib

In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
import os

print('All libraries imported successfully.')

In [ ]:

FRED_API_KEY = "2e22c2185be3bb846c2b2579a5f076f6"

# Crisis event dates
CRISES = {
    "9/11 (2001)":             "2001-09-11",
    "Iraq War (2003)":         "2003-03-20",
    "Financial Crisis (2008)": "2008-09-15",
    "COVID-19 (2020)":         "2020-03-11",
    "Russia-Ukraine (2022)":   "2022-02-24",
    "Israel-Hamas (2023)":     "2023-10-07",
}

# Financial assets to download from Yahoo Finance
TICKERS = {
    "SP500":      "^GSPC",
    "VIX":        "^VIX",
    "MSCI_World": "EFA",      
    "Defense":    "LMT",      
    "Energy":     "XLE",      
    "Health":     "XLV",      
    "Tech":       "XLK",      
    "Aviation_BA":  "BA",
    "Aviation_DAL": "DAL",     
    "Gold":       "GC=F",     
    "Oil":        "CL=F",     
}

# Macroeconomic series from FRED
FRED_SERIES = {
    "Fed_Rate": "FEDFUNDS",
    "CPI":      "CPIAUCSL",
    "Unemploy": "UNRATE",
    "M2":       "M2SL",
}

print('Settings ready.')

In [ ]:
# Step 1: Download financial data (yfinance)

print('Downloading financial data from Yahoo Finance...')

symbols = list(TICKERS.values())

raw = yf.download(
    tickers=symbols,
    start="2000-01-01",
    end="2025-01-01",
    progress=True
)

# Keep only closing prices
prices = raw["Close"].copy()

# Rename columns: "^GSPC" -> "SP500", "^VIX" -> "VIX", etc.
symbol_to_name = {v: k for k, v in TICKERS.items()}
prices.rename(columns=symbol_to_name, inplace=True)

# Drop rows where all values are missing (market holidays)
prices.dropna(how="all", inplace=True)

# Combine aviation: use DAL where available, BA for earlier dates
prices["Aviation"] = prices["Aviation_DAL"].fillna(prices["Aviation_BA"])
prices.drop(columns=["Aviation_DAL", "Aviation_BA"], inplace=True)

print(f'Done! {prices.shape[0]} trading days, {prices.shape[1]} assets')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
prices.head()

In [ ]:
# Step 2: Download macroeconomic data (FRED)
from fredapi import Fred

fred = Fred(api_key="2e22c2185be3bb846c2b2579a5f076f6")

FRED_SERIES = {
    "Fed_Rate": "FEDFUNDS",
    "CPI":      "CPIAUCSL",
    "Unemploy": "UNRATE",
    "M2":       "M2SL",
}

print("Downloading macroeconomic data from FRED...")

all_series = {}
for name, series_id in FRED_SERIES.items():
    print(f"  Downloading {name} ({series_id})...")
    series = fred.get_series(series_id, observation_start="2000-01-01", observation_end="2025-01-01")
    all_series[name] = series

macro = pd.DataFrame(all_series)
macro = macro.resample("D").ffill()
macro = macro.loc["2000-01-01":"2025-01-01"]

print(f"Done! {macro.shape[0]} days, {macro.shape[1]} indicators")
macro.head()

In [ ]:
# Step 3: Calculate daily returns
# Return = (today's price - yesterday's price) / yesterday's price * 100
# Calculate daily returns
returns = prices.pct_change() * 100
# Drop only rows where ALL values are missing, not just some
returns.dropna(how='all', inplace=True)

print(f'Returns calculated: {returns.shape[0]} days')
returns.head()

In [ ]:
# Step 4: Create crisis windows (30 days before and after each crisis)

windows = {}

for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    pre_start   = crisis_date - pd.Timedelta(days=30)
    post_end    = crisis_date + pd.Timedelta(days=30)

    windows[crisis_name] = {
        "crisis_date":  crisis_date,
        "pre_prices":   prices.loc[pre_start  : crisis_date - pd.Timedelta(days=1)],
        "post_prices":  prices.loc[crisis_date : post_end],
        "pre_returns":  returns.loc[pre_start  : crisis_date - pd.Timedelta(days=1)],
        "post_returns": returns.loc[crisis_date : post_end],
    }

    pre_len  = len(windows[crisis_name]["pre_returns"])
    post_len = len(windows[crisis_name]["post_returns"])
    print(f'{crisis_name}: {pre_len} days before, {post_len} days after')

In [ ]:
# Step 5: Plot S&P 500 with crisis dates marked

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(prices.index, prices["SP500"], color="#1f77b4", linewidth=1, label="S&P 500")

for crisis_name, crisis_date_str in CRISES.items():
    crisis_date = pd.Timestamp(crisis_date_str)
    ax.axvline(x=crisis_date, color="red", linestyle="--", alpha=0.6, linewidth=1)
    ax.text(crisis_date, prices["SP500"].max() * 0.92, crisis_name,
            rotation=90, fontsize=7, color="red", va="top")

ax.set_title("S&P 500 (2000-2025) — Crisis Dates Marked")
ax.set_xlabel("Date")
ax.set_ylabel("Closing Price ($)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Save all data to CSV files

# os.makedirs creates the 'data' folder automatically if it doesn't exist
save_dir = os.path.expanduser("~/Desktop/data")
os.makedirs(save_dir, exist_ok=True)

prices.to_csv(os.path.join(save_dir, "prices.csv"))
returns.to_csv(os.path.join(save_dir, "returns.csv"))
if macro is not None:
    macro.to_csv(os.path.join(save_dir, "macro.csv"))

for crisis_name, w in windows.items():
    safe_name = crisis_name.replace("/", "-").replace(" ", "_")
    w["pre_returns"].to_csv(os.path.join(save_dir, f"{safe_name}_pre.csv"))
    w["post_returns"].to_csv(os.path.join(save_dir, f"{safe_name}_post.csv"))

print("All files saved to:", save_dir)
for f in os.listdir(save_dir):
    print(f"  {f}")

## Summary

| Variable | What it contains |
|---|---|
| `prices` | Daily closing prices for all assets (2000–2025) |
| `returns` | Daily percentage change for all assets |
| `macro` | Interest rate, CPI, unemployment, money supply |
| `windows` | 30-day pre and post data for each crisis |


In [ ]:
print("Dataset overview:")
print(f"  - {prices.shape[0]} trading days, {prices.shape[1]} assets")
print(f"  - Date range: 2000-01-03 to 2024-12-31")
print(f"  - Aviation: combined BA (pre-2007) + DAL (post-2007)")
print(f"  - Macro indicators: Fed Rate, CPI, Unemployment, M2")
print(f"  - Crisis windows: {len(CRISES)} events × pre/post 30 days")